In [32]:
import pandas as pd

heart = pd.read_csv('heart.csv')

# Создаём новый признак old: 1, если возраст больше 60, иначе 0
heart['old'] = (heart['age'] > 60).astype(int)

# Посчитаем количество пациентов старше 60 лет
result = heart['old'].sum()

print(result)

79


Датасет болезней сердца содержит информацию о пациентах и переменную предсказания target — наличие у пациента болезни сердца.

Датасет содержит следующие признаки:

    age — возраст
    sex — пол (1 - мужчина, 0 - женщина)
    cp — тип боли в груди (4 значения)
    trestbps — артериальное давление в покое
    chol — холестерин сыворотки в мг/дл
    fbs — уровень сахара в крови натощак > 120 мг/дл
    restecg — результаты электрокардиографии в покое (значения 0,1,2)
    thalach — достигнута максимальная частота сердечных сокращений
    exang — стенокардия, вызванная физической нагрузкой
    oldpeak — депрессия ST, вызванная физической нагрузкой, по сравнению с состоянием покоя
    slope — наклон пикового сегмента ST при нагрузке
    ca — количество крупных сосудов (0-3), окрашенных при флюроскопии
    thal — дефект, где 3 = нормальный; 6 = фиксированный дефект; 7 = обратимый дефект

In [33]:
import numpy as np

# Создадим функцию для определения нормы давления по возрасту и полу
def get_trestbps_mean(age, sex):
    if age < 20:
        return 123 if sex == 1 else 116
    elif 21 <= age <= 30:
        return 126 if sex == 1 else 120
    elif 31 <= age <= 40:
        return 129 if sex == 1 else 127
    elif 41 <= age <= 50:
        return 135 if sex == 1 else 137
    elif 51 <= age <= 60:
        return 142 if sex == 1 else 144
    else:  # 61 и старше
        return 142 if sex == 1 else 159

# Применим функцию к датафрейму для создания нового признака
heart['trestbps_mean'] = heart.apply(lambda row: get_trestbps_mean(row['age'], row['sex']), axis=1)

# Выведем значение признака для пациента с номером 300
result = heart.loc[300, 'trestbps_mean']
print(result)


142


In [34]:
import category_encoders as ce

# Список признаков, требующих OneHotEncoding
categorical_cols = ['cp', 'restecg', 'slope', 'ca', 'thal']

# Инициализируем OneHotEncoder из библиотеки category_encoders
encoder = ce.OneHotEncoder(cols=categorical_cols)

# Применяем кодирование и удаляем исходные столбцы
heart_encoded = encoder.fit_transform(heart)

# Выводим количество признаков после кодирования
result = heart_encoded.shape[1]
print(result)


30


In [35]:
from sklearn.preprocessing import RobustScaler

# Выделяем числовые признаки
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'trestbps_mean']

# Инициализация RobustScaler
scaler = RobustScaler()

# Применяем RobustScaler ко всем числовым признакам
heart[numerical_cols] = scaler.fit_transform(heart[numerical_cols])

# Выводим стандартное отклонение признака chol после нормализации (округляем до 6 знаков)
result = round(heart['chol'].std(), 6)
print(result)


0.816232


In [36]:
import pandas as pd
import numpy as np

import category_encoders as ce

categorical_cols = ['cp', 'restecg', 'slope', 'ca', 'thal']
encoder = ce.OneHotEncoder(cols=categorical_cols)

# Преобразование и добавление новых закодированных признаков, исходные удаляются
heart_encoded = encoder.fit_transform(heart)

# Вычисляем корреляционную матрицу с абсолютными значениями
corr_matrix = heart.corr().abs()

# Список нужных пар признаков
pairs = [
    ('age', 'old'),
    ('age', 'trestbps_mean'),
    ('slope_1', 'slope_3'),
    ('thal_1', 'thal_2'),
    ('thal_3', 'thal_4'),
    ('cp_1', 'ca_1'),
    ('cp_1', 'cp_2'),
    ('thal_2', 'thal_3'),
    ('restecg_1', 'restecg_2'),
    ('slope_2', 'slope_3'),
    ('age', 'exang'),
    ('age', 'chol')
]

for col1, col2 in pairs:
    if col1 in heart_encoded.columns and col2 in heart_encoded.columns:
        corr_value = heart_encoded[col1].corr(heart_encoded[col2])
        print(f"Корреляция между {col1} и {col2}: {corr_value:.3f}")
    else:
        print(f"Столбец {col1} или {col2} отсутствует в данных")


Корреляция между age и old: 0.718
Корреляция между age и trestbps_mean: 0.763
Корреляция между slope_1 и slope_3: -0.253
Корреляция между thal_1 и thal_2: -0.277
Корреляция между thal_3 и thal_4: -0.065
Корреляция между cp_1 и ca_1: 0.069
Корреляция между cp_1 и cp_2: -0.182
Корреляция между thal_2 и thal_3: -0.873
Корреляция между restecg_1 и restecg_2: -0.974
Корреляция между slope_2 и slope_3: -0.870
Корреляция между age и exang: 0.097
Корреляция между age и chol: 0.214
